# Advanced tutorial: a small generic multi-reservoir configuration

This notebook reads `examples/multiple_reservoirs`. It explains IDs, topology,
method mixing, and state separation without executing a simulation or provider.

## Setup

In [1]:
from pathlib import Path
import html, json, re, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/hydrocnhs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from tutorials.build_notebooks import CMS_TO_CFS

class ReadableText:
    def __init__(self, title, text):
        self.title, self.text = title, text
    def _repr_markdown_(self):
        fence = "`" * max(4, 1 + max((len(run) for run in re.findall(r"`+", self.text)), default=0))
        return f"#### {self.title}\n\n{fence}text\n{self.text}\n{fence}"
    def _repr_html_(self):
        return f'<h4>{html.escape(self.title)}</h4><pre>{html.escape(self.text)}</pre>'

pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25,
                     "font.family": "Arial", "font.size": 11, "axes.titlesize": 12,
                     "axes.labelsize": 11, "legend.fontsize": 11})

## 1. Begin with complete maintained source files

The experiment delegates hydrology, physical network, agents, models,
information, auditors, and extensions to separate files. Paths are relative to
the file that declares them.

In [2]:
example = ROOT / "examples/multiple_reservoirs"
for title, name in (("Rule-only experiment", "experiment.yaml"),
                    ("Mixed-method experiment", "experiment.llm.yaml"),
                    ("Physical network", "reservoirs.yaml"), ("Hydrology bindings", "hydrology.yaml")):
    display(ReadableText(title, (example / name).read_text(encoding="utf-8")))

#### Rule-only experiment

````text
schema_version: hydroops-coupled-simulation.v2
run_id: independent-rivers-rule
output: ../../runs/independent-rivers-rule
case: reservoirs.yaml
start_date: '2020-02-28'
end_date: '2020-03-02'
max_calls: null
hydrology:
  file: hydrology.yaml
agents:
  file: agents.rule.yaml
models:
  file: models.yaml
information:
  file: information.yaml
auditors:
  file: auditors.yaml
extensions:
  local_retrieval: extensions/__init__.py
  hanasaki: ../rules/hanasaki/__init__.py

````

#### Mixed-method experiment

````text
schema_version: hydroops-coupled-simulation.v2
run_id: independent-rivers-mixed
output: ../../runs/independent-rivers-mixed
case: reservoirs.yaml
start_date: '2020-02-28'
end_date: '2020-03-02'
max_calls: null
hydrology:
  file: hydrology.yaml
agents:
  file: agents.llm.yaml
models:
  file: models.yaml
information:
  file: information.yaml
auditors:
  file: auditors.yaml
extensions:
  local_retrieval: extensions/__init__.py
  hanasaki: ../rules/hanasaki/__init__.py

````

#### Physical network

````text
schema_version: reservoir-physical-case.v1
data_kind: synthetic_fixture
network:
  schema_version: llm-reservoir-network.v2
  start_date: '2020-02-28'
  reservoirs:
    lake_a:
      inflow_node: inlet_a
      physical:
        reservoir_id: lake_a
        capacity_m3: 1000000.0
        dead_storage_m3: 0.0
        initial_storage_m3: 500000.0
        minimum_release_cms: 0.0
        maximum_release_cms: 20.0
        dt_seconds: 86400.0
    lake_b:
      inflow_node: inlet_b
      physical:
        reservoir_id: lake_b
        capacity_m3: 1000000.0
        dead_storage_m3: 0.0
        initial_storage_m3: 500000.0
        minimum_release_cms: 0.0
        maximum_release_cms: 20.0
        dt_seconds: 86400.0
  connections:
  - source: lake_a
    target: river_a
    unit_hydrograph:
    - 0.0
    - 1.0
  - source: lake_b
    target: river_b
    unit_hydrograph:
    - 0.0
    - 1.0
  outlets:
  - river_a
  - river_b

````

#### Hydrology bindings

````text
model: model.yaml
forcing: forcing.csv
zero_flow_nodes: []
inflow_bindings:
  inlet_a:
    runoff: basin_a
  inlet_b:
    runoff: basin_b
  river_a:
    constant: 0
  river_b:
    constant: 0

````

## 2. Keep node, reservoir, and agent IDs distinct

`inlet_a` and `inlet_b` are nodes. `lake_a` and `lake_b` are reservoir keys.
`operator_a` and `operator_b` are agents. Each agent's `reservoir_id` must match
one reservoir key, and its history belongs only to that reservoir.

In [3]:
import yaml
network = yaml.safe_load((example / "reservoirs.yaml").read_text(encoding="utf-8"))["network"]
agents = yaml.safe_load((example / "agents.llm.yaml").read_text(encoding="utf-8"))
display(pd.DataFrame([{"agent ID": a, "reservoir ID": s["reservoir_id"],
    "inflow node": network["reservoirs"][s["reservoir_id"]]["inflow_node"], "type": s["type"]}
    for a, s in agents.items()]))
display(ReadableText("Mixed Rule-based and LLM agents", (example / "agents.llm.yaml").read_text(encoding="utf-8")))

,agent ID,reservoir ID,inflow node,type
0,operator_a,lake_a,inlet_a,llm
1,operator_b,lake_b,inlet_b,rule_based


#### Mixed Rule-based and LLM agents

````text
operator_a:
  reservoir_id: lake_a
  type: llm
  model_profile: local
  prompt_file: prompts/operator.txt
  customization:
    system_prompt_file: prompts/system.txt
    prompt_version: '1'
    prompt_variables:
      purpose: demonstrate an independently configured operator
    context_providers:
      guidance:
        extension: local_retrieval
        config:
          documents: knowledge/guidance.json
        preview: 'Offline preview: use only the supplied state and local operating guidance.'
  auditors:
  - bounds
operator_b:
  reservoir_id: lake_b
  type: rule_based
  rule: hanasaki
  rule_parameters:
    purpose: flood_control
    alpha: 0.85
    inflow_statistics_file: statistics_b.json
  auditors:
  - bounds

````

## 3. Choose topology explicitly

The fixture has independent parallel branches: `inlet_a → lake_a → river_a`
and `inlet_b → lake_b → river_b`. A parallel confluence points both reservoirs
to one junction. A series layout sends an upstream reservoir to a downstream
reservoir inlet. Apply each local inflow once and justify every unit hydrograph.

In [4]:
display(pd.DataFrame([
    {"layout": "independent", "connections": "lake_a→river_a; lake_b→river_b", "outlets": "river_a, river_b"},
    {"layout": "parallel", "connections": "lake_a→junction; lake_b→junction", "outlets": "junction"},
    {"layout": "series", "connections": "lake_a→inlet_b; lake_b→outlet", "outlets": "outlet"}]))

,layout,connections,outlets
0,independent,lake_a→river_a; lake_b→river_b,"river_a, river_b"
1,parallel,lake_a→junction; lake_b→junction,junction
2,series,lake_a→inlet_b; lake_b→outlet,outlet


## 4. Mix methods without mixing state

The mixed file assigns an LLM to `lake_a` and a Rule-based operator to `lake_b`.
Separate agent IDs, reservoir IDs, histories, and outputs prevent one reservoir's
decision history from becoming another's state. A shared model profile does not
create shared conversation history.

In [5]:
display(pd.DataFrame([{"agent ID": a, "reservoir ID": s["reservoir_id"],
    "method": "LLM" if s["type"] == "llm" else "Rule-based",
    "local retrieval": bool(s.get("customization", {}).get("context_providers")),
    "state key": f"{a}:{s['reservoir_id']}"} for a, s in agents.items()]))

,agent ID,reservoir ID,method,local retrieval,state key
0,operator_a,lake_a,LLM,True,operator_a:lake_a
1,operator_b,lake_b,Rule-based,False,operator_b:lake_b


## 5. Adapt the fixture

Replace synthetic forcing, basin parameters, reservoir physics, topology,
routing, prompts, knowledge, model settings, and auditors with site evidence.
First check hydrology, units, mass balance, topology, and state isolation with an
offline or Rule-based operator; enable an LLM after the physical path is clear.

[Generic configuration](../docs/USER_GUIDE.md) · [HydroCNHS setup](../docs/HYDROLOGY_SETUP.md) ·
[Multiple-reservoir source files](../examples/multiple_reservoirs/)